# 06. 출퇴근 경로 → 100m 지점(터널 주변 25m), 주행 방위각, 터널 구간

출력: `data/2_중간/routes.pkl`(경로 선 + 터널 구간, 05의 chain과 같은 형식), `data/3_결과/route_points.csv`(100m 지점, 터널 주변 25m, `tunnel_zone` 포함), `data/3_결과/routes.png`(확인용 지도)

| 경로 | 출근 | 퇴근 |
|---|---|---|
| A 신천대로 | 북대구IC 부근 → 신천대로 남행 → 수성구 → 본점 | 역방향 |
| B 경부고속도로 | 칠곡·구미 방면 → 경부고속도로 → 동대구IC → 본점 | 역방향 |
| C 달구벌대로 | 달서구 성서 → 달구벌대로 동행 → 본점 | 역방향 |

**어떻게 경로를 그리나요?**
OSM 대구 도로망에서 최단경로를 찾되, 그 경로의 핵심 도로(신천대로 등)는 길이를 짧게 쳐서(가중치) 그 도로를 따라가도록 유도해요.
퇴근은 상·하행 차로가 다르니 도착지→출발지로 따로 경로를 구해요.
아래 **경로 확인** 셀에서 도로 이름 비율과 지도를 꼭 확인하세요. 이상하면 `ROUTES`의 좌표나 경유지를 고치면 됩니다.

In [ ]:
%pip install osmnx geopandas matplotlib -q

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import LineString
from shapely.ops import linemerge
from pyproj import Geod

ROOT = Path.cwd()
EXTERNAL, PROCESSED, RESULT = ROOT / "data" / "1_원본", ROOT / "data" / "2_중간", ROOT / "data" / "3_결과"
RESULT.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
GEOD, CRS_M = Geod(ellps="WGS84"), 5179
ox.settings.use_cache = True
ox.settings.cache_folder = str(PROCESSED / "osm_cache")

## 1. 출발지·경유지·도착지
`geocode`로 이름을 먼저 찾아보고, 실패하면 옆의 대략 좌표(위도, 경도)를 써요. 출력된 좌표가 맞는지 지도에서 확인해 주세요.

In [ ]:
def place(query, fallback):
    try:
        lat, lon = ox.geocode(query)
        src = "geocode"
    except Exception:
        lat, lon = fallback
        src = "대략좌표"
    print(f"  {query:<28} → ({lat:.5f}, {lon:.5f}) [{src}]")
    return (lat, lon)

print("지점 좌표")
HQ = place("대구광역시 수성구 달구벌대로 2310", (35.8573, 128.6306))       # iM뱅크 본점
P = {
    "북대구IC": place("북대구IC", (35.9046, 128.5922)),
    "칠곡": place("경상북도 칠곡군 왜관읍", (35.9900, 128.3980)),
    "동대구IC": place("동대구IC", (35.8835, 128.6440)),
    "성서": place("대구광역시 달서구 성서IC", (35.8436, 128.4977)),
}

# 경유지 목록, 우선 따라갈 도로 이름(가중치 ↓)
ROUTES = {
    "A_신천대로":     {"stops": [P["북대구IC"], HQ],             "prefer": ["신천대로"]},
    "B_경부고속도로": {"stops": [P["칠곡"], P["동대구IC"], HQ],  "prefer": ["경부고속도로"]},
    "C_달구벌대로":   {"stops": [P["성서"], HQ],                 "prefer": ["달구벌대로"]},
}
PREFER_WEIGHT = 0.3     # 선호 도로는 길이의 30%로 계산

## 2. 대구 도로망 받기 (처음 한 번, 몇 분)

In [ ]:
GRAPH_CACHE = PROCESSED / "daegu_drive.graphml"
if GRAPH_CACHE.exists():
    G = ox.load_graphml(GRAPH_CACHE)
else:
    lats = [p[0] for r in ROUTES.values() for p in r["stops"]]
    lons = [p[1] for r in ROUTES.values() for p in r["stops"]]
    bbox = (min(lons) - 0.03, min(lats) - 0.03, max(lons) + 0.03, max(lats) + 0.03)   # (서, 남, 동, 북)
    G = ox.graph_from_bbox(bbox, network_type="drive", simplify=True)
    ox.save_graphml(G, GRAPH_CACHE)
print(f"노드 {len(G.nodes):,}, 도로 조각 {len(G.edges):,}")

## 3. 경로 찾기 (출근 + 퇴근)

In [ ]:
def as_list(x):
    return x if isinstance(x, list) else [x]

def edge_attr(G, u, v, prefer):
    # u→v 사이 여러 도로 중 가중치가 가장 작은 것
    best = None
    for k, d in G[u][v].items():
        names = [str(n) for n in as_list(d.get("name", "")) if n == n]
        w = d["length"] * (PREFER_WEIGHT if any(p in n for p in prefer for n in names) else 1)
        if best is None or w < best[0]:
            best = (w, d)
    return best

def route_line(stops, prefer):
    for u, v, k, d in G.edges(keys=True, data=True):
        names = [str(n) for n in as_list(d.get("name", "")) if n == n]
        d["w"] = d["length"] * (PREFER_WEIGHT if any(p in n for p in prefer for n in names) else 1)
    nodes = [ox.distance.nearest_nodes(G, lon, lat) for lat, lon in stops]
    path = []
    for a, b in zip(nodes[:-1], nodes[1:]):
        seg = nx.shortest_path(G, a, b, weight="w")
        path += seg if not path else seg[1:]
    coords, tunnels, names, speeds, s = [], [], [], [], 0.0
    for u, v in zip(path[:-1], path[1:]):
        _, d = edge_attr(G, u, v, prefer)
        if "geometry" in d:
            pts = list(d["geometry"].coords)
        else:
            pts = [(G.nodes[u]["x"], G.nodes[u]["y"]), (G.nodes[v]["x"], G.nodes[v]["y"])]
        g = gpd.GeoSeries([LineString(pts)], crs=4326).to_crs(CRS_M).iloc[0]
        coords.extend(list(g.coords) if not coords else list(g.coords)[1:])
        is_tunnel = any(str(t) not in ("no", "nan", "None") for t in as_list(d.get("tunnel")) if t == t) \
                    or any(str(c) == "yes" for c in as_list(d.get("covered")))
        if is_tunnel:
            if tunnels and abs(tunnels[-1][1] - s) < 1:
                tunnels[-1][1] = s + g.length
            else:
                tunnels.append([s, s + g.length])
        name = "/".join(str(n) for n in as_list(d.get("name", "")) if n == n) or "(이름없음)"
        names.append((s, s + g.length, name))
        speeds.append(pd.to_numeric(as_list(d.get("maxspeed", np.nan))[0], errors="coerce"))
        s += g.length
    return LineString(coords), [tuple(t) for t in tunnels if t[1] - t[0] >= 30], names, np.nanmedian(speeds) if speeds else np.nan

rows, name_tables = [], {}
for rname, cfg in ROUTES.items():
    for trip, stops in [("출근", cfg["stops"]), ("퇴근", cfg["stops"][::-1])]:
        line, tunnels, names, speed = route_line(stops, cfg["prefer"])
        rid = f"{rname}_{trip}"
        rows.append({"route_id": rid, "route": rname, "trip": trip, "geometry": line, "tunnels": tunnels,
                     "maxspeed": speed if speed == speed else (100 if "고속" in rname else 60), "length_m": line.length})
        name_tables[rid] = names
routes = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_M)
routes[["route_id", "length_m", "maxspeed"]].assign(터널수=routes.tunnels.str.len(), length_km=lambda d: (d.length_m / 1000).round(1))

## 4. 경로 확인: 어떤 도로를 몇 km 지나는지

In [ ]:
for rid, names in name_tables.items():
    t = pd.DataFrame(names, columns=["s0", "s1", "도로"]).assign(km=lambda d: (d.s1 - d.s0) / 1000)
    print(f"\n[{rid}] 총 {t.km.sum():.1f} km")
    print(t.groupby("도로").km.sum().sort_values(ascending=False).head(6).round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"   # 윈도우 한글 폰트
ax = routes.to_crs(4326).plot(column="route_id", legend=True, figsize=(9, 9), linewidth=2)
gpd.GeoSeries(gpd.points_from_xy([HQ[1]], [HQ[0]]), crs=4326).plot(ax=ax, color="red", markersize=60)
ax.set_title("출퇴근 경로 (빨간 점 = 본점)")
plt.savefig(RESULT / "routes.png", dpi=120, bbox_inches="tight")

## 5. 경로 지점 + 실제 주행 방위각 + 터널 구간
- 기본은 100m 간격이에요. 100m 간격만 쓰면 짧은 터널이나 구간 경계를 건너뛸 수 있어서, **각 터널 입구 200m 앞부터 출구 150m 뒤까지는 25m 간격**으로 촘촘히 둬요.
- 구간 경계 지점은 꼭 넣어요: 입구 −150, 입구, +100, +400, 출구 −60, 출구, 출구 +150.
- `tunnel_zone`은 05와 같은 규칙과 이름(진입전150 / 경계100 / 이행300 / 기본부 / 출구60 / 진출후150 / 일반)이라서, 05로 학습한 결과를 그대로 적용할 수 있어요. 입구·출구까지 거리 열은 저장하지 않아요.
- 방위각은 각 지점 앞뒤 50m를 이은 방향이에요(북=0°, 동=90°).


In [ ]:
def sample_s(L, tunnels):
    s = set(np.round(np.arange(0, L, 100), 1))
    for t0, t1 in tunnels:
        s |= set(np.round(np.arange(t0 - 200, t1 + 150 + 1e-6, 25), 1))                        # 터널 주변 25m 간격
        s |= {round(x, 1) for x in [t0 - 150, t0, t0 + 100, t0 + 400, t1 - 60, t1, t1 + 150]}     # 구간 경계
    return sorted(x for x in s if 0 <= x <= L)

def zone_at(s, tunnels):                       # 05의 tunnel_feats와 같은 규칙
    for t0, t1 in tunnels:
        if t0 <= s <= t1:
            d_in, d_out = s - t0, t1 - s
            return "출구60" if d_out <= 60 else "경계100" if d_in <= 100 else "이행300" if d_in <= 400 else "기본부"
    nxt = [t0 - s for t0, _ in tunnels if t0 > s]
    prv = [s - t1 for _, t1 in tunnels if t1 < s]
    if nxt and min(nxt) <= 150:
        return "진입전150"
    if prv and min(prv) <= 150:
        return "진출후150"
    return "일반"

pts = []
for r in routes.itertuples():
    line = r.geometry
    for k, s in enumerate(sample_s(line.length, r.tunnels)):
        a, b = line.interpolate(max(s - 50, 0)), line.interpolate(min(s + 50, line.length))
        pts.append({"route_id": r.route_id, "route": r.route, "trip": r.trip, "seq": k, "s": s,
                    "p": line.interpolate(s), "a": a, "b": b,
                    "in_tunnel": any(t0 <= s <= t1 for t0, t1 in r.tunnels), "tunnel_zone": zone_at(s, r.tunnels)})
pts = pd.DataFrame(pts)
to_ll = lambda col: gpd.GeoSeries(pts[col], crs=CRS_M).to_crs(4326)
p, a, b = to_ll("p"), to_ll("a"), to_ll("b")
pts["lat"], pts["lon"] = p.y.to_numpy(), p.x.to_numpy()
pts["heading"] = GEOD.inv(a.x.to_numpy(), a.y.to_numpy(), b.x.to_numpy(), b.y.to_numpy())[0] % 360
road_at = {rid: pd.DataFrame(n, columns=["s0", "s1", "도로"]) for rid, n in name_tables.items()}
pts["도로"] = [road_at[r].loc[(road_at[r].s0 <= s), "도로"].iloc[-1] for r, s in zip(pts.route_id, pts.s)]
pts = pts.drop(columns=["p", "a", "b"])

routes.to_pickle(PROCESSED / "routes.pkl")
pts.to_csv(RESULT / "route_points.csv", index=False, encoding="utf-8-sig")
print(pts.groupby("route_id").agg(지점수=("seq", "size"), 터널지점=("in_tunnel", "sum")))
pd.crosstab(pts.route_id, pts.tunnel_zone)
